# SR Artifact Detection — end-to-end pipeline

Trains the artifact-segmentation U-Net, searches the decision threshold, saves
the weights, and visualises predictions. A short interpolation demo is included
first.

## Before you run

Add these course-provided files to the project root (from `additional_files.zip`):
`useful_utils.py`, `eval_metric.py`, `artifact_dataset.py`.

The dataset lives on disk and is **not** downloaded here. Set `DATASET_DIR` below
to the folder containing `labels.csv` (columns `sr_fn`, `mask_fn`, `gt_fn`,
`has_artifact`) and the images. On Colab you can mount Google Drive and point
`DATASET_DIR` at e.g. `/content/drive/MyDrive/SR-task/train`.

## 1. Interpolation kernels (quick demo)

Fully self-contained — no external files needed.

In [ ]:
import numpy as np
from interpolation.kernels import interpolate, nearest_neighbor_kernel, linear_kernel, cubic_kernel

signal = np.sin(np.linspace(0, 2 * np.pi, 8))
for name, k in [("nearest", nearest_neighbor_kernel), ("linear", linear_kernel), ("cubic", cubic_kernel)]:
    up = interpolate(signal, 2, k)
    print(f"{name:8s} 8 -> {len(up)} samples")

## 2. Artifact detection

### Configuration

In [ ]:
from pathlib import Path
import torch

DATASET_DIR = Path("/content/drive/MyDrive/SR-task/train")   # <-- set to your dataset
LABELS = DATASET_DIR / "labels.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 20
BATCH_SIZE = 2
USE_GT = True
print("Device:", device)

### Augmentations and dataloaders

Uses the provided `create_dataloader`. The horizontal/vertical flips and crop are applied jointly to `img`, `gt` (`image1`) and `mask`.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from artifact_dataset import create_dataloader   # provided module

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
extra = {"image1": "image", "mask": "mask"}

train_augs = A.Compose([
    A.RandomCrop(height=700, width=900, p=0.3),
    A.Resize(height=1024, width=768),
    A.HorizontalFlip(p=0.3),
    A.VerticalFlip(p=0.3),
    A.HueSaturationValue(20, 30, 20, p=0.5),
    A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),
    ToTensorV2(),
], additional_targets=extra)

val_augs = A.Compose([
    A.Normalize(mean=MEAN, std=STD, max_pixel_value=255.0),
    ToTensorV2(),
], additional_targets=extra)

train_loader, val_loader = create_dataloader(
    DATASET_DIR, LABELS, batch_size=BATCH_SIZE, val_size=0.2,
    random_state=42, num_workers=2, train_augs=train_augs, val_augs=val_augs)

### Model, loss, optimizer

In [ ]:
from artifacts.model import MyModel
from artifacts.losses import CombinedBCEJaccardLoss
from torch.optim.lr_scheduler import CosineAnnealingLR

model = MyModel(num_blocks=3, start_filters=32, use_gt=USE_GT, init=True).to(device)
criterion = CombinedBCEJaccardLoss(bce_weight=0.25, jaccard_weight=0.75).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

### Train

The best checkpoint (by `0.75·IoU + 0.25·F1`) is saved; the chosen threshold is stored on `model.threshold`.

In [ ]:
from artifacts.engine import train_model
from eval_metric import iou   # provided module

train_model(model, NUM_EPOCHS, train_loader, val_loader, device,
            criterion, optimizer, iou_fn=iou,
            best_model_path="my_model.pth", scheduler=scheduler)

### Save / reload weights

The saved `my_model.pth` is what you submit. Verify it reloads.

In [ ]:
model.save_weights("my_model.pth")
reloaded = MyModel(num_blocks=3, start_filters=32, use_gt=USE_GT).to(device)
reloaded.load_weights("my_model.pth", device)
reloaded.threshold = model.threshold
print("reloaded OK | threshold:", reloaded.threshold)

### Visualise predictions

Expects `<image>.png` plus `<image>@gt.png` (and `<image>@mask.png` for the binary overlay).

In [ ]:
from artifacts.inference import visualize_probability_mask, visualize_binary_mask

TEST_IMAGE = "test_image.png"   # <-- a test image with @gt.png and @mask.png siblings
visualize_probability_mask(reloaded, TEST_IMAGE, device)
visualize_binary_mask(reloaded, TEST_IMAGE, device)